In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scikit-digital-health

In [ ]:
import os
import glob
import traceback
import pandas as pd
import numpy as np
import skdh
from skdh.io import ReadCwa

In [ ]:
PD_INPUT_DIR   = "/content/drive/MyDrive/dataset/PD"
CONTROL_INPUT_DIR  = "/content/drive/MyDrive/dataset/Control"

OUTPUT_BASE    = "/content/drive/MyDrive/Lasige"
PD_OUTPUT_DIR  = os.path.join(OUTPUT_BASE, "PD")
CONTROL_OUTPUT_DIR = os.path.join(OUTPUT_BASE, "Control")

os.makedirs(PD_OUTPUT_DIR, exist_ok=True)
os.makedirs(CONTROL_OUTPUT_DIR, exist_ok=True)

PATIENTS_CSV = "/content/drive/MyDrive/dataset/patients.csv"
patients_activities = pd.read_csv(PATIENTS_CSV)

# FIX timezone: start_ts/end_ts vienen en epoch ms (UTC). El reloj del sensor
# (columna "Time" de raw_Trunk.csv) esta en hora local de Portugal, que puede
# ser UTC+0 (invierno) o UTC+1 (verano) segun la fecha de la sesion.
# Convertimos a Europe/Lisbon para que el offset correcto se aplique
# automaticamente en cada fecha, y luego quitamos el tz para comparar
# directamente con los datetimes "naive" de raw_Trunk.csv.
patients_activities["start_ts"] = (
    pd.to_datetime(patients_activities["start_ts"], unit="ms", utc=True)
    .dt.tz_convert("Europe/Lisbon")
    .dt.tz_localize(None)
)
patients_activities["end_ts"] = (
    pd.to_datetime(patients_activities["end_ts"], unit="ms", utc=True)
    .dt.tz_convert("Europe/Lisbon")
    .dt.tz_localize(None)
)

MAX_PATIENTS = 15
PATIENT_HEIGHT_M = 1.69


In [ ]:
gait = skdh.gait_old.Gait(
    correct_accel_orient=True,
    use_cwt_scale_relation=True,
    min_bout_time=6.0,
    max_bout_separation_time=0.25,
    max_stride_time=2.25,
    loading_factor=0.2,
    height_factor=0.53,
    prov_leg_length=False,
    filter_order=4,
    filter_cutoff=20.0,
    downsample_aa_filter=True,
    day_window=(0, 24),
)

In [ ]:
#Procura um ficheiro dentro da pasta do paciente que contenha uma palavra-chave.Ignora maiusculas e minusculas. Devolve o caminho completo do ficheiro ou none
def find_trunk_file(patient_dir, needle):
    for path in glob.glob(os.path.join(patient_dir, "*")):
        if needle.lower() in os.path.basename(path).lower():
            return path
    return None


#Obtem uma lista ordenada com os caminhos de todas as subpastas dentro de um diretorio raiz.

def list_patient_dirs(root_dir):
    return sorted(d for d in glob.glob(os.path.join(root_dir, "*")) if os.path.isdir(d))


#Executa a analise da marcha para um paciente e guarda os relatorios em CSV: um ficheiro com todas as metricas por ciclo de marcha e o resumo estatistico (medias e desvios-padrao).
#Os bouts vs actividad ja NO se guardan por paciente: se devuelven para combinarlos en un unico CSV general (ver process_pd / process_datos_control / celda final).
#bliblioteca velha

def run_gait_and_save(patient_id, t, acc, fs, output_dir, group,
                       recording_start=None, time_is_relative=True):
    results = gait.predict(time=t, accel=acc, fs=fs, height=PATIENT_HEIGHT_M)

    bout_n = results.get("Bout N", [])
    n_bouts = len(np.unique(bout_n)) if len(bout_n) > 0 else 0
    n_steps = len(results.get("IC Time", []))
    n_cycles = len(results.get("Gait Cycles", []))
    print(f"  Bouts: {n_bouts} | Steps: {n_steps} | Gait cycles: {n_cycles}")

    #Ver bouts contra actividad real
    bouts_df = get_unique_bouts(results)
    if not bouts_df.empty:
        if time_is_relative:
            if recording_start is None:
                raise ValueError(
                    "recording_start es None pero time_is_relative=True"
                )
            bouts_df["start_abs"] = (
                recording_start
                + pd.to_timedelta(bouts_df["start_s"], unit="s")
            )
        else:
            bouts_df["start_abs"] = pd.to_datetime(bouts_df["start_s"], unit="s")
        bouts_df["end_abs"] = bouts_df["start_abs"] + pd.to_timedelta(bouts_df["duration_s"], unit="s")

        labels = bouts_df.apply(
            lambda r: label_bout_activity(r["start_abs"], r["end_abs"], patient_id, patients_activities),
            axis=1, result_type="expand"
        )
        bouts_df[["matched_activity", "overlap_duration"]] = labels
        bouts_df["patient_id"] = patient_id

        n_walking = (bouts_df["matched_activity"] == "Walking slow").sum()
        print(f"  Bouts que caen en 'Walking slow': {n_walking}/{len(bouts_df)}")

#filtra resultas que representam as metricas(comecam com PARAM)
    gait_metrics = {k: v for k, v in results.items() if k.startswith("PARAM:")}
    metrics_df = pd.DataFrame(gait_metrics)  #Converte metricas num dataframe

#Resumo info do paciente
    summary_row = {
        "patient_id": patient_id,
        "Group": group,
        "n_bouts": n_bouts,
        "n_steps": n_steps,
        "n_gait_cycles": n_cycles,
    }
    for col in metrics_df.columns:
        summary_row[f"{col}_mean"] = metrics_df[col].mean()  #calcula media
        summary_row[f"{col}_std"] = metrics_df[col].std()    #e desvio padrao de cada metrica obtida
    summary_df = pd.DataFrame([summary_row])

#define caminhos para exportar os ficheiros csv (ja nao se guarda bouts_vs_activity aqui)
    cycles_path = os.path.join(output_dir, f"{patient_id}_all_gait_cycles.csv")
    summary_path = os.path.join(output_dir, f"{patient_id}_gait_summary.csv")

    metrics_df.to_csv(cycles_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    print(f"  Guardado -> {cycles_path}")
    print(f"  Guardado -> {summary_path}")

    return summary_df, bouts_df

#Comparacao dos patients.csv
def get_unique_bouts(results):
    bout_n = results.get("Bout N", np.array([]))
    if len(bout_n) == 0:
        return pd.DataFrame(columns=["bout_n", "start_s", "duration_s", "n_steps"])
    df = pd.DataFrame({
        "bout_n": bout_n,
        "start_s": results.get("Bout Starts", np.array([])),
        "duration_s": results.get("Bout Duration", np.array([])),
        "n_steps": results.get("Bout Steps", np.array([])),
    })
    return df.drop_duplicates(subset="bout_n").reset_index(drop=True)


def label_bout_activity(bout_start, bout_end, patient_id, patients_df):
    acts = patients_df[patients_df["code"] == patient_id]
    best_activity, best_overlap = None, pd.Timedelta(0)
    for _, row in acts.iterrows():
        overlap_start = max(bout_start, row["start_ts"])
        overlap_end = min(bout_end, row["end_ts"])
        overlap = overlap_end - overlap_start
        if overlap > best_overlap:
            best_overlap, best_activity = overlap, row["activity"]
    return best_activity, best_overlap


In [ ]:
# Processa o datased PD , Le os ficheiros dos pacientes, extrai o tempo e os dados do acelerometro,
 #   executa a analise de marcha e guarda um resumo consolidado de todos os pacientes num unico ficheiro CSV.
def process_pd():
    print("\n=== Processando dataset PD ===")
    all_summaries = []  #lista dos resumos ind de cada paciente
    all_bouts = []       #lista dos bouts_vs_activity de cada paciente (para o CSV geral)
    processed = 0

    for patient_dir in list_patient_dirs(PD_INPUT_DIR):
        if processed >= MAX_PATIENTS: #Interrompe com o max
            break

        patient_id = os.path.basename(patient_dir)
        csv_path = find_trunk_file(patient_dir, "raw_trunk")  #Procura ficheiro trunk
        if csv_path is None or not csv_path.lower().endswith(".csv"):   # Verifica se o ficheiro existe e se e csv.
            print(f"[{patient_id}] nao encontrou raw_Trunk.csv, salta-se")
            continue

        print(f"[{patient_id}] procesando {csv_path}")
        try:
            df = pd.read_csv(csv_path)
            df.columns = df.columns.str.strip() #remove espacos nos nomes das colunas

            t_datetime = pd.to_datetime(df["Time"])
            recording_start = t_datetime.iloc[0]

            t = (t_datetime - recording_start).dt.total_seconds().to_numpy()
              #converte para segundos

            acc = df[["Accel-X (g)", "Accel-Y (g)", "Accel-Z (g)"]].to_numpy()

            summary_df, bouts_df = run_gait_and_save(patient_id, t, acc, fs=None, output_dir=PD_OUTPUT_DIR, group="PD", recording_start=recording_start, time_is_relative=True )
            all_summaries.append(summary_df)    #Executa analise de marcha e guarda os resultados
            all_bouts.append(bouts_df)
            processed += 1

        except Exception as e:
            print(f"[{patient_id}] ERRO: {e}")  #Captura os erros do paciente se tiver
            traceback.print_exc()
#combina os resumos
    if all_summaries:
        combined = pd.concat(all_summaries, ignore_index=True)
        combined_path = os.path.join(PD_OUTPUT_DIR, "ALL_patients_summary.pd.csv")
        combined.to_csv(combined_path, index=False)
        print(f"\nResumo combinado PD guardado -> {combined_path}")

    print(f"PD: {processed} paciente(s) procesado(s)")
    return all_bouts


In [ ]:
#Control dados
#Extrai os dados temporais, aceleracao e frequencia de amostragem (fs),
#Executa a analise de marcha e guarda um resumo consolidado de todos os pacientes num unico ficheiro CSV.


def process_datos_control():
    print("\n=== Processando dataset Control ===")

    all_summaries = []
    all_bouts = []       #lista dos bouts_vs_activity de cada paciente (para o CSV geral)
    processed = 0

    for patient_dir in list_patient_dirs(CONTROL_INPUT_DIR):

        if processed >= MAX_PATIENTS:
            break

        patient_id = os.path.basename(patient_dir)

        # Buscar raw_Trunk.csv
        csv_path = find_trunk_file(patient_dir, "raw_trunk")

        if csv_path is None or not csv_path.lower().endswith(".csv"):
            print(f"[{patient_id}] nao  encontrou raw_Trunk.csv, skips")
            continue

        print(f"[{patient_id}] procesando {csv_path}")

        try:
            # Leer CSV
            df = pd.read_csv(csv_path)

            # Limpiar nombres de columnas
            df.columns = df.columns.str.strip()

            # Tiempo
            t_datetime = pd.to_datetime(df["Time"])

            recording_start = t_datetime.iloc[0]

            t = (
            (t_datetime - recording_start)
            .dt.total_seconds()
            .to_numpy()
            )

            # Aceleracion
            acc = df[
                [
                    "Accel-X (g)",
                    "Accel-Y (g)",
                    "Accel-Z (g)"
                ]
            ].to_numpy()

            # Analisis de marcha
            summary_df, bouts_df = run_gait_and_save(
                patient_id,
                t,
                acc,
                fs=None,
                output_dir=CONTROL_OUTPUT_DIR,
                group="Control",
                recording_start=recording_start,
                time_is_relative=True
            )

            all_summaries.append(summary_df)
            all_bouts.append(bouts_df)

            processed += 1

        except Exception as e:
            print(f"[{patient_id}] ERROR: {e}")
            traceback.print_exc()

    # Combinar resultados
    if all_summaries:

        combined = pd.concat(
            all_summaries,
            ignore_index=True
        )

        combined_path = os.path.join(
            CONTROL_OUTPUT_DIR,
            "ALL_patients_summary.control.csv"
        )

        combined.to_csv(
            combined_path,
            index=False
        )

        print(
            f"\nResumo combinado Control guardado -> "
            f"{combined_path}"
        )

    print(
        f"Control: {processed} paciente(s) processado(s)"
    )
    return all_bouts


In [ ]:
all_bouts_pd = process_pd()
all_bouts_control = process_datos_control()

# Combina los bouts_vs_activity de TODOS los pacientes (PD + Control) en un unico CSV general
all_bouts = [b for b in (all_bouts_pd + all_bouts_control) if b is not None and not b.empty]
if all_bouts:
    combined_bouts = pd.concat(all_bouts, ignore_index=True)
    combined_bouts_path = os.path.join(OUTPUT_BASE, "ALL_patients_bouts_vs_activity.csv")
    combined_bouts.to_csv(combined_bouts_path, index=False)
    print(f"\nResumo geral de bouts vs activity guardado -> {combined_bouts_path}")
else:
    print("\nNao ha bouts para combinar (lista vazia).")
